# Task 3: Citation Span Extraction - BERT QA (v2)

**Model:** bert-base-uncased (Question Answering)

**Changes from v1:**
- ✅ Fix `compute_metrics`: convert token positions → text via `offset_mapping` → so sánh text thật (character-level F1 + EM)
- ✅ Fix val tokenization: giữ lại `context`, `answer_text`, `offset_mapping` trong `val_references` để compute_metrics dùng
- ✅ Xóa cell training config trùng lặp (cell 16 và 17 trong v1)
- ✅ Xóa cell save model trùng lặp

**v1 issues:**
- ❌ `compute_metrics` dùng token-level overlap → F1 97% bị inflate
- ❌ Val dataset remove `context` → không extract được predicted text
- ❌ Hai cell training config trùng nhau

---

## 1. Setup & Imports

In [22]:
import transformers, datasets, accelerate
print(f"✅ transformers: {transformers.__version__}")
print(f"✅ datasets: {datasets.__version__}")
print(f"✅ accelerate: {accelerate.__version__}")

✅ transformers: 5.0.0
✅ datasets: 4.8.3
✅ accelerate: 1.12.0


## 2. Weights & Biases Setup

In [23]:
import wandb
from kaggle_secrets import UserSecretsClient

try:
    secrets = UserSecretsClient()
    key = secrets.get_secret("WANDB_API_KEY")

    result = wandb.login(key=key)

    if result:
        print("✅ Wandb logged in")
    else:
        print("⚠️ Wandb login returned False - API key có thể sai")

except Exception as e:
    print(f"⚠️ Wandb login failed: {type(e).__name__}: {e}")

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: WARNING [wandb.login()] Changing session credentials to explicit value for https://api.wandb.ai.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc


✅ Wandb logged in


## 3. Check Dataset

In [24]:
import os

train_path = "/kaggle/input/datasets/tathiyennhi/task3-citation-span-extraction/task3/train"
val_path = "/kaggle/input/datasets/tathiyennhi/task3-citation-span-extraction/task3/val"

train_count = len([f for f in os.listdir(train_path) if f.endswith('.label')])
val_count = len([f for f in os.listdir(val_path) if f.endswith('.label')])

print(f"✅ Train: {train_count:,} files")
print(f"✅ Val: {val_count:,} files")

✅ Train: 54,356 files
✅ Val: 2,910 files


## 4. Load Data

In [25]:
import json
from pathlib import Path
from datasets import Dataset

def load_task3_data(data_dir, max_examples=None):
    """
    Load data and return as list (for non-streaming dataset)
    """
    data_path = Path(data_dir)
    label_files = sorted(data_path.glob("*.label"))
    
    if max_examples:
        label_files = label_files[:max_examples]
    
    total_files = len(label_files)
    print(f"📊 Loading {total_files:,} files from {data_dir}")
    
    examples = []
    skipped = 0

    for i, label_file in enumerate(label_files):
        if (i+1) % 5000 == 0:
            print(f"⏳ {i+1:,}/{total_files:,} | Loaded: {len(examples):,} | Skipped: {skipped}")

        try:
            with open(label_file) as f:
                label_data = json.load(f)
        except:
            skipped += 1
            continue

        text = label_data.get('text', '')
        if not text:
            skipped += 1
            continue
            
        citation_spans = label_data.get('citation_spans', [])

        for span_info in citation_spans:
            citation_id = span_info.get('citation_id', '')
            span_text = span_info.get('span_text', '')
            s_span = span_info.get('s_span', -1)
            e_span = span_info.get('e_span', -1)
            
            if s_span == -1 or e_span == -1 or s_span >= e_span:
                skipped += 1
                continue

            question = f"What does citation {citation_id} support?"
            
            examples.append({
                'question': question,
                'context': text,
                'answer_text': span_text,
                'answer_start_char': s_span,
                'answer_end_char': e_span
            })

    print(f"✅ Loaded {len(examples):,} examples | Skipped: {skipped}")
    return examples

# Load data
print("=" * 60)
train_examples = load_task3_data(train_path)
val_examples = load_task3_data(val_path)

# Convert to Dataset
train_dataset = Dataset.from_list(train_examples)
val_dataset = Dataset.from_list(val_examples)

print(f"\n✅ Train dataset: {len(train_dataset):,} examples")
print(f"✅ Val dataset: {len(val_dataset):,} examples")

📊 Loading 54,356 files from /kaggle/input/datasets/tathiyennhi/task3-citation-span-extraction/task3/train
⏳ 5,000/54,356 | Loaded: 11,791 | Skipped: 0
⏳ 10,000/54,356 | Loaded: 23,469 | Skipped: 0
⏳ 15,000/54,356 | Loaded: 34,981 | Skipped: 0
⏳ 20,000/54,356 | Loaded: 46,313 | Skipped: 0
⏳ 25,000/54,356 | Loaded: 57,749 | Skipped: 0
⏳ 30,000/54,356 | Loaded: 69,736 | Skipped: 0
⏳ 35,000/54,356 | Loaded: 81,576 | Skipped: 0
⏳ 40,000/54,356 | Loaded: 93,307 | Skipped: 0
⏳ 45,000/54,356 | Loaded: 104,943 | Skipped: 0
⏳ 50,000/54,356 | Loaded: 116,406 | Skipped: 0
✅ Loaded 126,622 examples | Skipped: 0
📊 Loading 2,910 files from /kaggle/input/datasets/tathiyennhi/task3-citation-span-extraction/task3/val
✅ Loaded 6,894 examples | Skipped: 0

✅ Train dataset: 126,622 examples
✅ Val dataset: 6,894 examples


## 5. Tokenization

In [26]:
from transformers import AutoTokenizer

MODEL_PATH = "bert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
print(f"✅ Tokenizer loaded from: {MODEL_PATH}")


def prepare_features(examples):
    tokenized = tokenizer(
        examples["question"],
        examples["context"],
        max_length=512,
        truncation="only_second",
        padding="max_length",
        return_offsets_mapping=True,
    )

    start_positions = []
    end_positions = []

    for i in range(len(examples["question"])):
        start_char = examples["answer_start_char"][i]
        end_char = examples["answer_end_char"][i]

        offsets = tokenized["offset_mapping"][i]
        sequence_ids = tokenized.sequence_ids(i)

        context_start = 0
        while sequence_ids[context_start] != 1:
            context_start += 1

        context_end = len(sequence_ids) - 1
        while sequence_ids[context_end] != 1:
            context_end -= 1

        if not (offsets[context_start][0] <= start_char and offsets[context_end][1] >= end_char):
            start_positions.append(0)
            end_positions.append(0)
            continue

        start_token = context_start
        for idx in range(context_start, context_end + 1):
            if offsets[idx][0] <= start_char < offsets[idx][1]:
                start_token = idx
                break

        end_token = context_end
        for idx in range(context_start, context_end + 1):
            if offsets[idx][0] < end_char <= offsets[idx][1]:
                end_token = idx
                break

        start_positions.append(start_token)
        end_positions.append(end_token)

    tokenized["start_positions"] = start_positions
    tokenized["end_positions"] = end_positions

    return tokenized


print("Tokenizing train dataset...")
train_dataset = train_dataset.map(
    prepare_features,
    batched=True,
    remove_columns=["question", "context", "answer_text", "answer_start_char", "answer_end_char"],
)

# Val: giữ lại context, answer_text, offset_mapping để dùng trong compute_metrics
print("Tokenizing val dataset...")
val_dataset_tokenized = val_dataset.map(
    prepare_features,
    batched=True,
    remove_columns=["question", "answer_start_char", "answer_end_char"],
)

# Lưu references (context + answer_text + offset_mapping) riêng
val_references = [
    {
        "context": val_dataset_tokenized[i]["context"],
        "answer_text": val_dataset_tokenized[i]["answer_text"],
        "offset_mapping": val_dataset_tokenized[i]["offset_mapping"],
    }
    for i in range(len(val_dataset_tokenized))
]

# Xóa khỏi dataset trước khi đưa vào Trainer (model không cần)
val_dataset = val_dataset_tokenized.remove_columns(["context", "answer_text", "offset_mapping"])

print(f"\n✅ Tokenization complete")
print(f"   Train: {len(train_dataset):,} examples")
print(f"   Val:   {len(val_dataset):,} examples | val_references: {len(val_references):,}")

✅ Tokenizer loaded from: bert-base-uncased
Tokenizing train dataset...


Map:   0%|          | 0/126622 [00:00<?, ? examples/s]

Tokenizing val dataset...


Map:   0%|          | 0/6894 [00:00<?, ? examples/s]


✅ Tokenization complete
   Train: 126,622 examples
   Val:   6,894 examples | val_references: 6,894


## 6. Character-Level Metrics

In [27]:
import numpy as np
import re
from collections import Counter


def normalize_text(s):
    s = s.lower().strip()
    s = re.sub(r"[^\w\s]", "", s)
    s = " ".join(s.split())
    return s


def compute_f1_text(pred_text, true_text):
    pred_tokens = normalize_text(pred_text).split()
    true_tokens = normalize_text(true_text).split()

    if len(pred_tokens) == 0 or len(true_tokens) == 0:
        return 0.0

    common = Counter(pred_tokens) & Counter(true_tokens)
    num_same = sum(common.values())

    if num_same == 0:
        return 0.0

    precision = num_same / len(pred_tokens)
    recall = num_same / len(true_tokens)
    return 2 * precision * recall / (precision + recall)


def compute_metrics(pred):
    """
    Character-level metrics:
    1. Convert predicted token positions → char positions via offset_mapping
    2. Extract predicted text từ context
    3. So sánh với answer_text thật → F1 + EM
    """
    start_logits, end_logits = pred.predictions
    start_preds = np.argmax(start_logits, axis=1)
    end_preds = np.argmax(end_logits, axis=1)

    # label_ids shape: (N, 2) → [:, 0] = start, [:, 1] = end
    if isinstance(pred.label_ids, tuple):
        start_labels = pred.label_ids[0]
        end_labels = pred.label_ids[1]
    else:
        start_labels = pred.label_ids[:, 0]
        end_labels = pred.label_ids[:, 1]

    exact_match = 0
    f1_total = 0.0
    total = len(start_preds)

    for i in range(total):
        ref = val_references[i]
        context = ref["context"]
        answer_text = ref["answer_text"]
        offset_mapping = ref["offset_mapping"]

        start_tok = int(start_preds[i])
        end_tok = int(end_preds[i])

        # Invalid span → empty prediction
        if start_tok > end_tok or start_tok >= len(offset_mapping) or end_tok >= len(offset_mapping):
            pred_text = ""
        else:
            char_start = offset_mapping[start_tok][0]
            char_end = offset_mapping[end_tok][1]
            pred_text = context[char_start:char_end]

        em = int(normalize_text(pred_text) == normalize_text(answer_text))
        exact_match += em
        f1_total += compute_f1_text(pred_text, answer_text)

    return {
        "exact_match": exact_match / total,
        "f1": f1_total / total,
    }


print("✅ Character-level metrics defined (v2 fix)")

✅ Character-level metrics defined (v2 fix)


## 7. Model Setup

In [28]:
from transformers import AutoModelForQuestionAnswering

# MODEL_PATH defined in tokenization cell above
model = AutoModelForQuestionAnswering.from_pretrained(MODEL_PATH)
print(f"✅ Model loaded from: {MODEL_PATH}")

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

BertForQuestionAnswering LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
qa_outputs.bias                            | MISSING    | 
qa_outputs.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized beca

✅ Model loaded from: bert-base-uncased


## 8. Training Configuration

In [29]:
from transformers import (
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
    EarlyStoppingCallback,
    TrainerCallback,
)
from pathlib import Path
from kaggle_secrets import UserSecretsClient
import wandb

WANDB_PROJECT = "task3-citation-span-extraction"
ARTIFACT_NAME = "task3-bert-checkpoints"
CHECKPOINT_DIR = "/kaggle/working/checkpoints/task3_bert"


# ─── Login wandb from Kaggle Secrets ──────────────────────────────────────────
try:
    secrets = UserSecretsClient()
    wandb_key = secrets.get_secret("WANDB_API_KEY")
    wandb.login(key=wandb_key)

    if wandb.run is None:
        wandb.init(
            project=WANDB_PROJECT,
            name="bert-base-lr3e5-batch32",
            config={
                "model": "bert-base-uncased",
                "task": "span-extraction-qa",
                "learning_rate": 3e-5,
                "batch_size": 32,
                "max_steps": 5000,
                "warmup_steps": 500,
                "weight_decay": 0.01,
            },
            resume="allow",
        )

    report_to = "wandb"
    print("✅ Wandb logged in and initialized")

except Exception as e:
    print(f"⚠️ Wandb not available: {type(e).__name__}: {e}")
    report_to = "none"


# ─── Auto-resume: download latest checkpoint from wandb artifact ─────────────
def download_wandb_checkpoint(
    project,
    artifact_name,
    download_dir="/kaggle/working/resume_ckpt",
):
    if report_to == "none":
        return None

    try:
        api = wandb.Api()
        artifact = api.artifact(
            f"{project}/{artifact_name}:latest",
            type="checkpoint",
        )
        path = artifact.download(root=download_dir)

        checkpoints = sorted(
            Path(path).glob("checkpoint-*"),
            key=lambda x: int(x.name.split("-")[-1]),
        )

        if checkpoints:
            print(f"⬇️ Downloaded: {checkpoints[-1].name} from wandb artifact")
            return str(checkpoints[-1])

        return path

    except Exception as e:
        print(f"ℹ️ No wandb artifact to resume from: {e}")
        return None


resume_checkpoint = download_wandb_checkpoint(WANDB_PROJECT, ARTIFACT_NAME)

if not resume_checkpoint:
    print("🆕 Starting fresh training")


# ─── Callback: upload each checkpoint to wandb artifact right after saving ───
class WandbCheckpointCallback(TrainerCallback):
    def on_save(self, args, state, control, **kwargs):
        if report_to == "none" or wandb.run is None:
            return

        ckpt_dir = Path(args.output_dir) / f"checkpoint-{state.global_step}"
        if not ckpt_dir.exists():
            return

        artifact = wandb.Artifact(
            name=ARTIFACT_NAME,
            type="checkpoint",
            metadata={
                "step": state.global_step,
                "best_metric": state.best_metric,
            },
        )
        artifact.add_dir(str(ckpt_dir))
        wandb.log_artifact(artifact)

        print(f"⬆️ checkpoint-{state.global_step} uploaded to wandb artifact")


# ─── Training config ──────────────────────────────────────────────────────────
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

training_args = TrainingArguments(
    output_dir=CHECKPOINT_DIR,
    max_steps=5000,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=4,
    learning_rate=3e-5,
    weight_decay=0.01,
    warmup_steps=500,
    eval_strategy="steps",
    eval_steps=500,
    logging_steps=100,
    save_strategy="steps",
    save_steps=500,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    fp16=True,
    report_to=report_to,
    seed=42,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[
        EarlyStoppingCallback(early_stopping_patience=3),
        WandbCheckpointCallback(),
    ],
)

print("\n💡 Training Configuration:")
print("   Model:           BERT-base-uncased")
print("   Max steps:       5,000")
print("   Effective batch: 32 (8 × 4)")
print("   Learning rate:   3e-5")
print("   Save/eval every: 500 steps → auto-upload to wandb")
print(f"   Resume from:     {resume_checkpoint or 'scratch'}")

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: WARNING [wandb.login()] Changing session credentials to explicit value for https://api.wandb.ai.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc


✅ Wandb logged in and initialized


wandb: Downloading large artifact 'task3-bert-checkpoints:latest', 1247.08MB. 10 files...
wandb:   10 of 10 files downloaded.  
Done. 00:00:02.9 (426.8MB/s)



💡 Training Configuration:
   Model:           BERT-base-uncased
   Max steps:       5,000
   Effective batch: 32 (8 × 4)
   Learning rate:   3e-5
   Save/eval every: 500 steps → auto-upload to wandb
   Resume from:     /kaggle/working/resume_ckpt


## 9. Training Model

In [30]:
print("=" * 60)
print("🚀 TRAINING BERT FOR CITATION SPAN EXTRACTION")
print("=" * 60)

trainer.train(resume_from_checkpoint=resume_checkpoint)

print("\n✅ Training complete!")

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

🚀 TRAINING BERT FOR CITATION SPAN EXTRACTION


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Step,Training Loss,Validation Loss


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

wandb: Adding directory to artifact (/kaggle/working/checkpoints/task3_bert/checkpoint-5008)... Done. 2.8s
Could not locate the best model at /kaggle/working/checkpoints/task3_bert/checkpoint-3500/pytorch_model.bin, if you are running a distributed training on multiple nodes, you should activate `--save_on_each_node`.


⬆️ checkpoint-5008 uploaded to wandb artifact

✅ Training complete!


## 10. Evaluate

In [31]:
print("📊 VALIDATION RESULTS")
print("=" * 60)

eval_results = trainer.evaluate()

for key, value in eval_results.items():
    if isinstance(value, float):
        print(f"{key}: {value:.4f}")
    else:
        print(f"{key}: {value}")

print("=" * 60)
print(f"\n✅ F1 Score: {eval_results.get('eval_f1', 0):.2%}")
print(f"✅ Exact Match: {eval_results.get('eval_exact_match', 0):.2%}")

📊 VALIDATION RESULTS


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


eval_loss: 0.1951
eval_exact_match: 0.9095
eval_f1: 0.9555
eval_runtime: 150.9156
eval_samples_per_second: 45.6810
eval_steps_per_second: 1.4310
epoch: 2.5307

✅ F1 Score: 95.55%
✅ Exact Match: 90.95%


## 11. Save Model

In [32]:
SAVE_DIR = "/kaggle/working/task3_bert_final_v2"

import os
os.makedirs(SAVE_DIR, exist_ok=True)

trainer.save_model(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)

print(f"✅ Model saved to: {SAVE_DIR}")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Model saved to: /kaggle/working/task3_bert_final_v2


## 12. Test Inference

In [39]:
import torch
from transformers import pipeline
SAVE_DIR = "/kaggle/working/task3_bert_final_v2"
qa_pipeline = pipeline(
    'question-answering',
    model=SAVE_DIR,
    tokenizer=SAVE_DIR,
    device=0 if torch.cuda.is_available() else -1
)

# Test example
test_context = "The original Delphi study was run in the 1950s and 1960s by the RAND corporation to help the US Government determine the nuclear capabilities of the Soviet Union [CITATION_1] [CITATION_2] . They were studying the unknown military futures market by asking a variety of experts to answer a battery of questions. The answers were collated and then distributed back to the experts for additional rounds of answering the same questions -but critically, with the collective opinions of the other experts to aid their synthesis."
test_question = "Which text is supported by [CITATION_2]?"

result = qa_pipeline(
    question=test_question,
    context=test_context
)

print("\n📋 Test Inference:")
print(f"Question: {test_question}")
print(f"Context: {test_context}")
print(f"\nPredicted Answer: {result['answer']}")
print(f"Confidence: {result['score']:.4f}")
print(f"Start: {result['start']}, End: {result['end']}")

print("\n✅ BERT TRAINING COMPLETE!")

# Finish wandb run
try:
    wandb.finish()
except:
    pass

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]


📋 Test Inference:
Question: Which text is supported by [CITATION_2]?
Context: The original Delphi study was run in the 1950s and 1960s by the RAND corporation to help the US Government determine the nuclear capabilities of the Soviet Union [CITATION_1] [CITATION_2] . They were studying the unknown military futures market by asking a variety of experts to answer a battery of questions. The answers were collated and then distributed back to the experts for additional rounds of answering the same questions -but critically, with the collective opinions of the other experts to aid their synthesis.

Predicted Answer: The
Confidence: 0.0000
Start: 0, End: 3

✅ BERT TRAINING COMPLETE!


In [40]:
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForQuestionAnswering

SAVE_DIR = "/kaggle/working/task3_bert_final_v2"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Load trained model/tokenizer
tokenizer = AutoTokenizer.from_pretrained(SAVE_DIR)
model = AutoModelForQuestionAnswering.from_pretrained(SAVE_DIR)
model.to(DEVICE)
model.eval()

# Replace these with a REAL training example from your processed dataset
question = "What statement is supported by [CITATION_2]?"
context = (
    "The original Delphi study was run in the 1950s and 1960s by the RAND "
    "corporation to help the US Government determine the nuclear capabilities "
    "of the Soviet Union [CITATION_1] [CITATION_2]."
)
gold_answer = "to help the US Government determine the nuclear capabilities of the Soviet Union"

# Tokenize
inputs = tokenizer(
    question,
    context,
    return_offsets_mapping=True,
    return_tensors="pt",
    truncation=True,
    max_length=384
)

offset_mapping = inputs.pop("offset_mapping")[0]
inputs = {k: v.to(DEVICE) for k, v in inputs.items()}

with torch.no_grad():
    outputs = model(**inputs)

start_logits = outputs.start_logits[0]
end_logits = outputs.end_logits[0]

start_probs = F.softmax(start_logits, dim=0)
end_probs = F.softmax(end_logits, dim=0)

# Top 10 start/end tokens
input_ids = inputs["input_ids"][0]
tokens = tokenizer.convert_ids_to_tokens(input_ids)

print("=== TOP START TOKENS ===")
top_start = torch.topk(start_probs, 10)
for score, idx in zip(top_start.values.tolist(), top_start.indices.tolist()):
    print(f"idx={idx:3d} | prob={score:.6f} | token={tokens[idx]}")

print("\n=== TOP END TOKENS ===")
top_end = torch.topk(end_probs, 10)
for score, idx in zip(top_end.values.tolist(), top_end.indices.tolist()):
    print(f"idx={idx:3d} | prob={score:.6f} | token={tokens[idx]}")

# Brute-force best valid span
best_score = -1
best_start = 0
best_end = 0
max_answer_len = 40

for s in range(len(start_probs)):
    for e in range(s, min(s + max_answer_len, len(end_probs))):
        score = start_probs[s].item() * end_probs[e].item()
        if score > best_score:
            best_score = score
            best_start = s
            best_end = e

pred_ids = input_ids[best_start:best_end + 1]
pred_text = tokenizer.decode(pred_ids, skip_special_tokens=True)

print("\n=== PREDICTION ===")
print("Question:", question)
print("Gold Answer:", gold_answer)
print("Predicted:", pred_text)
print("Score:", best_score)
print("Start idx:", best_start, "| End idx:", best_end)

# Show character span in original context
sequence_ids = inputs["input_ids"][0].new_zeros(len(offset_mapping))
# We reconstruct sequence ids by re-tokenizing with tokenizer call that supports sequence_ids
encoded = tokenizer(
    question,
    context,
    return_offsets_mapping=True,
    truncation=True,
    max_length=384
)
sequence_ids = encoded.sequence_ids()

context_start_char = None
context_end_char = None

if sequence_ids[best_start] == 1 and sequence_ids[best_end] == 1:
    context_start_char = offset_mapping[best_start][0].item()
    context_end_char = offset_mapping[best_end][1].item()
    print("Predicted char span:", context_start_char, context_end_char)
    print("Predicted context slice:", context[context_start_char:context_end_char])

print("\n✅ Debug done.")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

=== TOP START TOKENS ===
idx= 13 | prob=0.997194 | token=the
idx=  0 | prob=0.002738 | token=[CLS]
idx= 52 | prob=0.000008 | token=.
idx= 15 | prob=0.000008 | token=del
idx= 14 | prob=0.000005 | token=original
idx= 18 | prob=0.000004 | token=was
idx= 29 | prob=0.000003 | token=to
idx= 26 | prob=0.000003 | token=the
idx= 32 | prob=0.000003 | token=us
idx= 27 | prob=0.000002 | token=rand

=== TOP END TOKENS ===
idx= 52 | prob=0.989135 | token=.
idx= 51 | prob=0.005662 | token=]
idx=  0 | prob=0.005101 | token=[CLS]
idx= 13 | prob=0.000048 | token=the
idx= 46 | prob=0.000005 | token=]
idx= 41 | prob=0.000004 | token=union
idx= 17 | prob=0.000003 | token=study
idx= 23 | prob=0.000003 | token=and
idx= 14 | prob=0.000002 | token=original
idx= 29 | prob=0.000002 | token=to

=== PREDICTION ===
Question: What statement is supported by [CITATION_2]?
Gold Answer: to help the US Government determine the nuclear capabilities of the Soviet Union
Predicted: the original delphi study was run in the 19

In [41]:
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForQuestionAnswering

SAVE_DIR = "/kaggle/working/task3_bert_final_v2"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer = AutoTokenizer.from_pretrained(SAVE_DIR)
model = AutoModelForQuestionAnswering.from_pretrained(SAVE_DIR)

model.to(DEVICE)
model.eval()

# ── Real data từ val/done/72.label ────────────────────────────────
context = (
    "In order to identify determinants of Z. bailii tolerance to acetic acid "
    "at the genome level we used in this study a genomic library previously "
    "prepared from the highly acetic acid tolerant strain ISA1307, an "
    "interspecies hybrid between Z. bailii and a closely related species "
    "which was isolated from a continuous production plant of sparkling "
    "wine [CITATION_1] [CITATION_2] . This genomic library was used to rescue "
    "the high susceptibility phenotype of S. cerevisiae BY4741_haa1Δ. This "
    "mutant, deleted for HAA1 gene, was chosen due to its very high "
    "susceptibility to acetic acid to avoid the use of the much higher "
    "concentrations required to inhibit the parental strain growth. With "
    "this approach we also expected to identify the functional homologue of "
    "S. cerevisiae HAA1 gene in this hybrid strain. During the development "
    "of this study, our laboratory carried out the genome sequencing, "
    "assembly and annotation of ISA1307 [CITATION_3] . This hybrid strain "
    "has been on the focus of several physiological studies, some of them "
    "aiming at the understanding of the mechanisms underlying its "
    "remarkable intrinsic resistance to acetic acid. Differently from "
    "S. cerevisiae, the Z.bailii-derived hybrid strain ISA1307 co-consumes "
    "glucose and acetic acid when cultivated in glucose medium supplemented "
    "with a sublethal growth inhibitory concentration of acetic acid "
    "[CITATION_4] [CITATION_5] . Quantitative proteomic studies on the "
    "adaptive response of this strain indicate that in glucose and acetic "
    "acid cultures the acid is channelled through the TCA cycle "
    "[CITATION_6] . After glucose exhaustion, acetic acid being present as "
    "the sole carbon source, the content of several proteins involved in "
    "gluconeogenesis and pentose phosphate pathway was however found to "
    "increase [CITATION_7] ."
)

test_cases = [
    {
        "citation_id": "[CITATION_1]",
        "question": "What does citation [CITATION_1] support?",
        "gold_span_text": (
            "an interspecies hybrid between Z. bailii and a closely related "
            "species which was isolated from a continuous production plant "
            "of sparkling wine [CITATION_1] [CITATION_2] ."
        ),
        "s_span": 205,
        "e_span": 375,
    },
    {
        "citation_id": "[CITATION_3]",
        "question": "What does citation [CITATION_3] support?",
        "gold_span_text": (
            "During the development of this study, our laboratory carried out "
            "the genome sequencing, assembly and annotation of ISA1307 "
            "[CITATION_3] ."
        ),
        "s_span": 801,
        "e_span": 938,
    },
    {
        "citation_id": "[CITATION_6]",
        "question": "What does citation [CITATION_6] support?",
        "gold_span_text": (
            "Quantitative proteomic studies on the adaptive response of this "
            "strain indicate that in glucose and acetic acid cultures the "
            "acid is channelled through the TCA cycle [CITATION_6] ."
        ),
        "s_span": 1386,
        "e_span": 1566,
    },
]


def predict_span(question, context):
    inputs = tokenizer(
        question,
        context,
        return_offsets_mapping=True,
        return_tensors="pt",
        truncation=True,
        max_length=512,
    )

    encoded = tokenizer(
        question,
        context,
        return_offsets_mapping=True,
        truncation=True,
        max_length=512,
    )

    sequence_ids = encoded.sequence_ids()
    offset_mapping = inputs.pop("offset_mapping")[0]
    inputs = {k: v.to(DEVICE) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)

    start_probs = F.softmax(outputs.start_logits[0], dim=0)
    end_probs = F.softmax(outputs.end_logits[0], dim=0)

    best_score = -1
    best_start = 0
    best_end = 0

    for s in range(len(start_probs)):
        for e in range(s, min(s + 60, len(end_probs))):
            if sequence_ids[s] != 1 or sequence_ids[e] != 1:
                continue

            score = start_probs[s].item() * end_probs[e].item()

            if score > best_score:
                best_score = score
                best_start = s
                best_end = e

    char_start = offset_mapping[best_start][0].item()
    char_end = offset_mapping[best_end][1].item()

    pred_text = context[char_start:char_end]

    return pred_text, best_score, char_start, char_end


# ── Run tests ────────────────────────────────
for tc in test_cases:
    pred_text, score, cs, ce = predict_span(tc["question"], context)

    print("\n" + "=" * 60)
    print(f"Citation:  {tc['citation_id']}")
    print(f"Gold:      {tc['gold_span_text']}")
    print(f"Predicted: {pred_text}")
    print(f"Score:     {score:.4f}")
    print(f"Char span: [{cs}, {ce}] | Gold: [{tc['s_span']}, {tc['e_span']}]")

    match = tc["s_span"] == cs and tc["e_span"] == ce
    print(f"Exact match: {'✅' if match else '❌'}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]


Citation:  [CITATION_1]
Gold:      an interspecies hybrid between Z. bailii and a closely related species which was isolated from a continuous production plant of sparkling wine [CITATION_1] [CITATION_2] .
Predicted: bailii and a closely related species which was isolated from a continuous production plant of sparkling wine [CITATION_1] [CITATION_2] .
Score:     0.1903
Char span: [239, 375] | Gold: [205, 375]
Exact match: ❌

Citation:  [CITATION_3]
Gold:      During the development of this study, our laboratory carried out the genome sequencing, assembly and annotation of ISA1307 [CITATION_3] .
Predicted: During the development of this study, our laboratory carried out the genome sequencing, assembly and annotation of ISA1307 [CITATION_3] .
Score:     0.9952
Char span: [801, 938] | Gold: [801, 938]
Exact match: ✅

Citation:  [CITATION_6]
Gold:      Quantitative proteomic studies on the adaptive response of this strain indicate that in glucose and acetic acid cultures the acid is chann